# autoresearch-medimage - Colab autoresearch session

This notebook runs a real multi-experiment ChestX-ray14 search session in Colab.

What it does:
- uses a named `--session`
- runs a curated sequence of fixed-budget recipes
- auto-marks each run as `keep`, `candidate`, or `discard`
- syncs `results.tsv`, run logs, and checkpoints back to Drive
- finalizes the best run at the end

What it is not:
- not a code-mutating overnight agent loop
- not a single-baseline demo

For a single ChestX-ray14 baseline, use `notebooks/colab_chexnet.ipynb`.
For the fastest smoke test and BYOD flow, use `notebooks/colab_quickstart.ipynb`.

## 1. Setup

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Download the public repo snapshot into /content
!rm -rf /content/autoresearch-medimage /content/autoresearch-medimage-main /content/repo.zip
!wget -q https://github.com/mlungrengmail/autoresearch-medimage/archive/refs/heads/main.zip -O /content/repo.zip
!unzip -q /content/repo.zip -d /content
!mv /content/autoresearch-medimage-main /content/autoresearch-medimage
%cd /content/autoresearch-medimage

In [ ]:
# Install dependencies
!uv sync

## 2. Mount Drive and define the session

This notebook expects that you already have a ChestX-ray14 copy in Drive, for example under `MyDrive/chestxray14`.

The default path stages the dataset onto local `/content` SSD once, then reuses it for the whole session. That first copy takes time, but it makes repeated experiments much faster than training through the Drive mount.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
from datetime import datetime

# Update these before running the next cell.
CHESTXRAY14_SOURCE = '/content/drive/MyDrive/chestxray14'
SESSION = f"chestxray14_colab_search_{datetime.now():%Y%m%d_%H%M%S}"
RUNS_ROOT = '/content/drive/MyDrive/autoresearch_runs'
USE_LOCAL_SSD = True
MIN_IMPROVEMENT = 0.10

print('SESSION =', SESSION)

In [ ]:
import os
import shutil
import tarfile
import zipfile
from pathlib import Path

source = Path(CHESTXRAY14_SOURCE)
repo_data = Path('/content/autoresearch-medimage/data/chestxray14')
cache_dir = Path('/content/chestxray14_cache')

if not source.exists():
    raise FileNotFoundError(source)

if repo_data.is_symlink() or repo_data.exists():
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    else:
        repo_data.unlink()

target = source
if USE_LOCAL_SSD or source.is_file():
    if cache_dir.exists():
        print('Reusing existing local cache:', cache_dir)
    else:
        cache_dir.mkdir(parents=True, exist_ok=True)
        if source.is_dir():
            print('Copying ChestX-ray14 from Drive to local SSD. This can take a while on the first run...')
            shutil.copytree(source, cache_dir, dirs_exist_ok=True)
        elif source.suffix.lower() == '.zip':
            with zipfile.ZipFile(source) as archive:
                archive.extractall(cache_dir)
        elif source.name.endswith(('.tar.gz', '.tgz', '.tar')):
            with tarfile.open(source) as archive:
                archive.extractall(cache_dir)
        else:
            raise ValueError('Use a folder, .zip, .tar.gz, .tgz, or .tar archive.')
    target = cache_dir

os.symlink(target, repo_data, target_is_directory=True)
print('linked:', repo_data)
print('points to:', os.readlink(repo_data))

## 3. Prepare dataset metadata for this runtime

Always use `--refresh-splits` here. It rebuilds `splits.csv` with Colab/Linux paths and avoids stale Windows paths from previous local runs.

In [ ]:
!uv run prepare.py --dataset chestxray14 --refresh-splits

## 4. Search helpers

The helper below runs one recipe, compares it against the best kept run in the same session, marks the latest row, and syncs the session state to Drive.

In [ ]:
import shlex
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_DIR = Path('/content/autoresearch-medimage')
RUNS_DIR = Path(RUNS_ROOT) / SESSION
RUNLOG_DIR = REPO_DIR / 'runlogs'
RUNLOG_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def session_frame():
    results_path = REPO_DIR / 'results.tsv'
    if not results_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(results_path, sep='\t')
    view = df[df['dataset'].astype(str) == 'chestxray14'].copy()
    if 'session' in view.columns:
        view = view[view['session'].fillna('').astype(str) == SESSION].copy()
    if 'metric' in view.columns:
        view['metric'] = pd.to_numeric(view['metric'], errors='coerce')
    return view

def best_keep_metric():
    view = session_frame()
    if view.empty or 'status' not in view.columns:
        return None
    keep = view[view['status'].astype(str) == 'keep']
    if keep.empty:
        return None
    return float(keep['metric'].max())

def sync_session_state():
    shutil.copy2(REPO_DIR / 'results.tsv', RUNS_DIR / 'results.tsv')
    ckpt_src = REPO_DIR / 'checkpoints'
    if ckpt_src.exists():
        ckpt_dst = RUNS_DIR / 'checkpoints'
        if ckpt_dst.exists():
            shutil.rmtree(ckpt_dst)
        shutil.copytree(ckpt_src, ckpt_dst)
    log_src = REPO_DIR / 'runlogs'
    if log_src.exists():
        log_dst = RUNS_DIR / 'runlogs'
        if log_dst.exists():
            shutil.rmtree(log_dst)
        shutil.copytree(log_src, log_dst)

def run_logged(command, log_path):
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    with open(log_path, 'w', encoding='utf-8') as log_handle:
        proc = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
            log_handle.write(line)
        returncode = proc.wait()
    if returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}: {command}')

def mark_latest(status, note=''):
    command = [
        'uv', 'run', 'python', 'tools/mark_result.py',
        '--dataset', 'chestxray14',
        '--session', SESSION,
        '--status', status,
    ]
    if note:
        command.extend(['--append-note', note])
    subprocess.run(command, cwd=REPO_DIR, check=True)

def run_search_entry(label, args):
    before = best_keep_metric()
    log_path = RUNLOG_DIR / f"{int(time.time())}_{label}.log"
    command = ['uv', 'run', 'train.py', '--dataset', 'chestxray14', '--session', SESSION, '--save-checkpoint'] + list(args)
    run_logged(command, log_path)
    latest = session_frame().iloc[-1]
    metric = float(latest['metric'])
    if before is None:
        status = 'keep'
        note = 'session baseline'
    elif metric >= before + MIN_IMPROVEMENT:
        status = 'keep'
        note = f'improved by {metric - before:.3f}'
    elif metric >= before:
        status = 'candidate'
        note = f'within {MIN_IMPROVEMENT:.2f} of best keep'
    else:
        status = 'discard'
        note = f'below best keep by {before - metric:.3f}'
    mark_latest(status, note)
    sync_session_state()
    latest = session_frame().iloc[-1]
    print('\nDecision:', status, '| metric =', metric)
    return latest

def restore_session_from_drive(existing_session):
    source_dir = Path(RUNS_ROOT) / existing_session
    if not source_dir.exists():
        raise FileNotFoundError(source_dir)
    if (source_dir / 'results.tsv').exists():
        shutil.copy2(source_dir / 'results.tsv', REPO_DIR / 'results.tsv')
    if (source_dir / 'checkpoints').exists():
        dst = REPO_DIR / 'checkpoints'
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(source_dir / 'checkpoints', dst)
    if (source_dir / 'runlogs').exists():
        dst = REPO_DIR / 'runlogs'
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(source_dir / 'runlogs', dst)
    print('restored session from', source_dir)


## 5. Optional: restore a previous session after a runtime reset

In [ ]:
# Set this only if you want to resume a previous Drive-backed session.
RESTORE_SESSION = ''
if RESTORE_SESSION:
    restore_session_from_drive(RESTORE_SESSION)

## 6. Curated search plan

This is a Colab-friendly recipe-space search. Edit the plan if you want a shorter or longer session.

The default plan mixes:
- the canonical CheXNet baseline
- a stronger DenseNet variant
- three T4-friendly alternate backbones


In [ ]:
SEARCH_PLAN = [
    {'label': 'baseline_densenet121', 'args': ['--recipe', 'chexnet']},
    {'label': 'advanced_densenet121', 'args': ['--recipe', 'chexnet_advanced']},
    {'label': 'resnet50', 'args': ['--recipe', 'chexnet_resnet50']},
    {'label': 'efficientnet_b0', 'args': ['--recipe', 'chexnet_efficientnet_b0']},
    {'label': 'convnext_tiny', 'args': ['--recipe', 'chexnet_convnext_tiny']},
    # Optional heavier run if you want to try a medical foundation model:
    # {'label': 'biomedclip', 'args': ['--recipe', 'chexnet_biomedclip']},
]

SEARCH_PLAN

In [ ]:
results = []
for entry in SEARCH_PLAN:
    print('\n' + '=' * 80)
    print('RUNNING', entry['label'])
    print('=' * 80)
    results.append(run_search_entry(entry['label'], entry['args']))

display(session_frame()[['timestamp', 'recipe', 'backbone', 'metric', 'status']])

In [ ]:
# Review the session leaderboard at any point
view = session_frame().sort_values('metric', ascending=False)
display(view[['timestamp', 'recipe', 'backbone', 'metric', 'status', 'checkpoint_path']])

## 7. Finalize the winning run

This generates the progress plot, held-out test evaluation, figures, and paper draft for the best kept run in the session.

In [ ]:
!uv run python tools/finalize_run.py --dataset chestxray14 --session "{SESSION}"

In [ ]:
# Back up the final artifacts to Drive
final_dir = Path(RUNS_ROOT) / SESSION / 'final_artifacts'
final_dir.mkdir(parents=True, exist_ok=True)
for candidate in [
    REPO_DIR / 'results.tsv',
    REPO_DIR / 'progress.png',
]:
    if candidate.exists():
        shutil.copy2(candidate, final_dir / candidate.name)
for folder_name in ['checkpoints', 'runlogs']:
    src = REPO_DIR / folder_name
    if src.exists():
        dst = final_dir / folder_name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
paper_generated = REPO_DIR / 'paper' / 'generated'
if paper_generated.exists():
    dst = final_dir / 'paper_generated'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(paper_generated, dst)
print('Backed up final artifacts to', final_dir)